# RMIFO Quick Start - Framework Modularizado
## Análise de Resiliência de Redes de Fibra Ótica com PySpark

Este notebook demonstra o uso completo do framework RMIFO para:
- 📥 Carregar datasets de topologia de rede
- 📊 Processar e analisar grafos com GraphFrames
- ⚡ Simular falhas e quantificar impactos
- 📈 Gerar relatórios de criticidade

**Estrutura modularizada** permitindo fácil extensão e customização.

## 1️⃣ Importar Bibliotecas Necessárias

In [ ]:
import sys
sys.path.insert(0, '/workspaces/RMIFO/global-network-resilience')

# Imports do framework RMIFO
from src import (
    get_config, DataIngestManager, GraphProcessor, FailureSimulator,
    setup_logging, get_logger, Timer, generate_report_header,
    export_dataframe_to_parquet, print_dataframe_stats
)

# Imports padrão Python
from pyspark.sql import SparkSession
import logging
import pandas as pd
from pathlib import Path

# Configurar logging
setup_logging(level=logging.INFO)
logger = get_logger(__name__)

logger.info("✓ Todas as bibliotecas importadas com sucesso!")

## 2️⃣ Carregar Configuração Centralizada

O framework usa um ConfigManager que centraliza todas as configurações:

In [ ]:
# Carrega configuração do arquivo YAML
config_file = "config/main_config.yaml"
config = get_config(config_file)

# Exibe configurações principais
print(f"""
{'='*60}
CONFIGURAÇÃO DO PROJETO
{'='*60}
Spark App: {config.spark_config.app_name}
Master: {config.spark_config.master}
Memória: {config.spark_config.memory}

Diretório base: {config.data_config.base_path}
Formato CAIDA: {config.data_config.caida_format}
Formato cabos: {config.data_config.cables_format}

PageRank iterações: {config.graph_config.pagerank_iterations}
Criticality threshold: {config.analysis_config.criticality_threshold}
Seed de simulação: {config.simulation_config.seed}
{'='*60}
""")


## 3️⃣ Inicializar SparkSession

Configura e inicializa uma SparkSession com os parâmetros do projeto:

In [ ]:
# Inicializa SparkSession com configurações
spark_config = config.spark_config

spark = SparkSession.builder \
    .appName(spark_config.app_name) \
    .master(spark_config.master) \
    .config("spark.driver.memory", spark_config.memory) \
    .config("spark.sql.shuffle.partitions", spark_config.shuffle_partitions) \
    .getOrCreate()

logger.info(f"✓ SparkSession '{spark_config.app_name}' inicializado")
logger.info(f"  Master: {spark_config.master}")
logger.info(f"  Memória: {spark_config.memory}")


## 4️⃣ Carregamento de Datasets com DataIngestManager

O `DataIngestManager` oferece uma interface unificada para carregar dados:

In [ ]:
# Inicializa gerenciador de ingestão de dados
data_manager = DataIngestManager(spark, config)

# Carrega dados (use_samples=True carrega dados de exemplo para testes)
with Timer("Carregamento de datasets"):
    datasets = data_manager.load_all_data(use_samples=True)

topology_df = datasets['topology']
cables_df = datasets['cables']
geoip_df = datasets['geoip']

# Exibe informações dos datasets
print(f"""
{'='*60}
DATASETS CARREGADOS
{'='*60}
Topologia: {topology_df.count()} nós
Cabos: {cables_df.count()} cabos
GeoIP: {geoip_df.count() if geoip_df else 'N/A'} registros
{'='*60}
""")

# Amostra de dados de topologia
print("\n📊 Amostra de Topologia (5 primeiras linhas):")
topology_df.show(5, truncate=False)

# Amostra de dados de cabos
print("\n🔗 Amostra de Cabos (5 primeiras linhas):")
cables_df.show(5, truncate=False)


## 5️⃣ Construção e Análise de Grafos com GraphProcessor

O `GraphProcessor` constrói grafos usando GraphFrames e computa métricas:

In [ ]:
# Inicializa processador de grafos
graph_processor = GraphProcessor(spark, config)

# Constrói grafo a partir de topologia e cabos
with Timer("Construcción del grafo"):
    network_graph = graph_processor.create_network_graph(topology_df, cables_df)

logger.info(f"✓ Grafo construído com sucesso!")

# Executa análise completa
print("\n🔍 Iniciando análise de rede...")
with Timer("Análise completa de rede"):
    analysis_results = graph_processor.analyze_network(network_graph)

# Extrai resultados
basic_metrics = analysis_results['basic_metrics']
pagerank_df = analysis_results['pagerank']
components_df = analysis_results['components']
critical_nodes_df = analysis_results['critical_nodes']

# Exibe métricas básicas
print(f"""
{'='*60}
MÉTRICAS DA REDE
{'='*60}
Vértices (nós): {basic_metrics['num_vertices']}
Arestas (cabos): {basic_metrics['num_edges']}
Grau médio entrada: {basic_metrics['avg_in_degree']:.2f}
Grau médio saída: {basic_metrics['avg_out_degree']:.2f}
Densidade do grafo: {basic_metrics['density']:.4f}
{'='*60}
""")

# Exibe top 10 nós por PageRank
print("\n🎯 Top 10 Nós por PageRank (Importância):")
pagerank_df.orderBy("pagerank", ascending=False).show(10, truncate=False)

# Exibe nós críticos identificados
print(f"\n⚠️ Nós Críticos ({critical_nodes_df.count()} encontrados):")
critical_nodes_df.show(10, truncate=False)


## 6️⃣ Simulação de Falhas com FailureSimulator

O `FailureSimulator` quantifica o impacto de falhas de nós e arestas:

In [ ]:
# Inicializa simulador de falhas
failure_simulator = FailureSimulator(spark, config)
failure_simulator.set_base_graph(network_graph)

# Simula falhas aleatórias de nós
print(f"\n{'='*60}")
print("SIMULAÇÃO DE FALHAS DE NÓ")
print(f"{'='*60}")

with Timer("10 simulações de falha de nó"):
    node_impacts = failure_simulator.batch_simulate_random_failures(
        num_scenarios=10,
        failure_type="node"
    )

# Exibe relatório de impactos de nó
print(f"\nResumo de impactos (nós):")
for i, impact in enumerate(node_impacts[:5], 1):
    print(f"\n{i}. {impact.scenario}")
    print(f"   • Nós afetados: {impact.nodes_affected}")
    print(f"   • Arestas afetadas: {impact.edges_affected}")
    print(f"   • Perda de conectividade: {impact.connectivity_loss:.2f}%")
    print(f"   • Aumento de latência: {impact.avg_latency_increase:.2f}%")
    print(f"   • Componentes criadas: {impact.components_created}")

# Simula falhas aleatórias de arestas
print(f"\n{'='*60}")
print("SIMULAÇÃO DE FALHAS DE ARESTA (CABOS)")
print(f"{'='*60}")

with Timer("10 simulações de falha de aresta"):
    edge_impacts = failure_simulator.batch_simulate_random_failures(
        num_scenarios=10,
        failure_type="edge"
    )

# Exibe relatório de impactos de aresta
print(f"\nResumo de impactos (arestas):")
for i, impact in enumerate(edge_impacts[:5], 1):
    print(f"\n{i}. {impact.scenario}")
    print(f"   • Nós afetados: {impact.nodes_affected}")
    print(f"   • Arestas afetadas: {impact.edges_affected}")
    print(f"   • Perda de conectividade: {impact.connectivity_loss:.2f}%")
    print(f"   • Aumento de latência: {impact.avg_latency_increase:.2f}%")


## 7️⃣ Exportação de Resultados

Salvar resultados em múltiplos formatos:

In [ ]:
# Criar diretório de resultados
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print("EXPORTAÇÃO DE RESULTADOS")
print(f"{'='*60}")

# Exportar DataFrames principais
with Timer("Exportação de topologia"):
    export_dataframe_to_parquet(topology_df, str(output_dir / "topology"))

with Timer("Exportação de nós críticos"):
    export_dataframe_to_parquet(critical_nodes_df, str(output_dir / "critical_nodes"))

with Timer("Exportação de PageRank"):
    export_dataframe_to_parquet(pagerank_df, str(output_dir / "pagerank"))

# Exportar análise em JSON
analysis_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'metrics': {
        'num_vertices': int(basic_metrics['num_vertices']),
        'num_edges': int(basic_metrics['num_edges']),
        'avg_degree': float(basic_metrics['avg_in_degree']),
        'density': float(basic_metrics['density'])
    },
    'simulation': {
        'node_failures': len(node_impacts),
        'edge_failures': len(edge_impacts),
        'avg_connectivity_loss_nodes': float(sum(i.connectivity_loss for i in node_impacts) / len(node_impacts)) if node_impacts else 0,
        'avg_connectivity_loss_edges': float(sum(i.connectivity_loss for i in edge_impacts) / len(edge_impacts)) if edge_impacts else 0
    }
}

from src import export_results_to_json
with Timer("Exportação de resumo de análise"):
    export_results_to_json(analysis_summary, str(output_dir / "analysis_summary.json"))

print(f"\n✓ Resultados exportados em: {output_dir.absolute()}")
print(f"  • topology/ (Parquet)")
print(f"  • critical_nodes/ (Parquet)")
print(f"  • pagerank/ (Parquet)")
print(f"  • analysis_summary.json")


## 8️⃣ Exemplos de Uso Avançado

### Simular falha de um nó específico:

In [ ]:
# Exemplo 1: Simular falha de um nó específico (ID = 0)
print(f"\n{'='*60}")
print("EXEMPLO 1: Falha de Nó Específico (ID=0)")
print(f"{'='*60}")

degraded_graph_1, impact_1 = failure_simulator.simulate_node_failure(node_id=0)

print(f"""
Cenário: {impact_1.scenario}
Nós afetados: {impact_1.nodes_affected}
Arestas removidas: {impact_1.edges_affected}
Perda de conectividade: {impact_1.connectivity_loss:.2f}%
Aumento de latência: {impact_1.avg_latency_increase:.2f}%
Componentes desconectadas criadas: {impact_1.components_created}
""")

# Exemplo 2: Simular falha de aresta específica
print(f"{'='*60}")
print("EXEMPLO 2: Falha de Aresta Específica (0 -> 1)")
print(f"{'='*60}")

degraded_graph_2, impact_2 = failure_simulator.simulate_edge_failure(src=0, dst=1)

print(f"""
Cenário: {impact_2.scenario}
Nós afetados: {impact_2.nodes_affected}
Arestas afetadas: {impact_2.edges_affected}
Perda de conectividade: {impact_2.connectivity_loss:.2f}%
Aumento de latência: {impact_2.avg_latency_increase:.2f}%
""")

# Exemplo 3: Análise comparativa - grafo original vs degradado
print(f"{'='*60}")
print("EXEMPLO 3: Análise Comparativa")
print(f"{'='*60}")

resultado = f"""
Grafo Original:
  • Vértices: {network_graph.vertices.count()}
  • Arestas: {network_graph.edges.count()}

Após falha do nó 0:
  • Vértices: {degraded_graph_1.vertices.count()}
  • Arestas: {degraded_graph_1.edges.count()}
  
Diferença:
  • Vértices removidos: {network_graph.vertices.count() - degraded_graph_1.vertices.count()}
  • Arestas removidas: {network_graph.edges.count() - degraded_graph_1.edges.count()}
"""
print(resultado)


## ✅ Resumo & Próximos Passos

### O que foi demonstrado:

1. ✓ **Carregamento de dados** com `DataIngestManager`
2. ✓ **Construção de grafos** com GraphFrames
3. ✓ **Análise de rede** (PageRank, componentes, nós críticos)
4. ✓ **Simulação de falhas** (nós e arestas)
5. ✓ **Quantificação de impactos** (latência, conectividade)
6. ✓ **Exportação de resultados** (Parquet, JSON)

### Estrutura Modularizada:
- **config.py** → Configurações centralizadas
- **data_ingestion.py** → Carregamento de dados
- **graph_processor.py** → Processamento com GraphFrames
- **failure_simulator.py** → Simulação de falhas
- **utils.py** → Funções auxiliares
- **main.py** → Pipeline orquestrado

### Próximos passos:

1. **Adicionar dados reais**: Substitua `use_samples=True` por diretórios com dados CAIDA/TeleGeography
2. **Customizar análisess**: Estenda `GraphAnalyzer` com seus próprios algoritmos
3. **Escalabilidade**: Configure para cluster Spark em `config/main_config.yaml`
4. **Visualização**: Integre matplotlib/folium para mapas geográficos
5. **ML**: Desenvolva modelos para prever criticidade baseado em padrões

### Recursos:
- 📖 [README_USAGE.md](../README_USAGE.md) - Documentação completa
- ⚙️ [config/main_config.yaml](../config/main_config.yaml) - Configurações
- 🚀 [main.py](../main.py) - Pipeline completo
- 📚 [src/](../src/) - Código modularizado